In [1]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
data_path = "/kaggle/input/datasets/mikeelio4/multilabel-tags-merged/multilabel_tags_merged.csv"


df = pd.read_csv(data_path)



In [3]:
df = df[["Title", "Tags"]].copy()
df = df.dropna(subset=["Title", "Tags"])

df["Title"] = df["Title"].astype(str).str.strip()
df["Tags"] = df["Tags"].astype(str).str.strip()

df = df[(df["Title"] != "") & (df["Tags"] != "")]



In [4]:
df["label_list"] = df["Tags"].apply(lambda x: x.split())
df[["Title", "Tags", "label_list"]].head()  

,Title,Tags,label_list
0,# + items .append is not a function,javascript,[javascript]
1,# - how to parallel code that lock several obj...,c#,[c#]
2,# . what do and # do in this code,javascript,[javascript]
3,# .dialog is not a function error,javascript,[javascript]
4,# .dialog is not a function error after using ...,javascript,[javascript]


In [5]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 50
y shape: (1866458, 50)
First labels: ['active-directory' 'algorithm' 'amazon-ec2' 'android' 'apache' 'api'
 'architecture' 'bash' 'c#' 'c++' 'centos' 'data-structures'
 'database-design' 'debugging' 'design-patterns' 'dns' 'ftp' 'git' 'http'
 'image-processing']


In [6]:
X = df["Title"].tolist()

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", len(X_train), y_train.shape)
print("Val  :", len(X_val), y_val.shape)
print("Test :", len(X_test), y_test.shape)

Train: 1493166 (1493166, 50)
Val  : 186646 (186646, 50)
Test : 186646 (186646, 50)


In [8]:
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_size = min(650000, len(X_train))
val_size = min(50000, len(X_val))
test_size = min(50000, len(X_test))

train_idx = np.random.RandomState(42).choice(len(X_train), train_size, replace=False)
val_idx = np.random.RandomState(42).choice(len(X_val), val_size, replace=False)
test_idx = np.random.RandomState(42).choice(len(X_test), test_size, replace=False)

train_df = pd.DataFrame({
    "title": X_train.iloc[train_idx].values if hasattr(X_train, "iloc") else np.array(X_train)[train_idx],
    "labels": list(y_train[train_idx])
})

val_df = pd.DataFrame({
    "title": X_val.iloc[val_idx].values if hasattr(X_val, "iloc") else np.array(X_val)[val_idx],
    "labels": list(y_val[val_idx])
})

test_df = pd.DataFrame({
    "title": X_test.iloc[test_idx].values if hasattr(X_test, "iloc") else np.array(X_test)[test_idx],
    "labels": list(y_test[test_idx])
})
print(y_train.dtype, y_val.dtype, y_test.dtype)

float32 float32 float32


In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['title', 'labels'],
    num_rows: 650000
})
Dataset({
    features: ['title', 'labels'],
    num_rows: 50000
})
Dataset({
    features: ['title', 'labels'],
    num_rows: 50000
})


In [10]:
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["title"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 50


In [13]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=120,
    per_device_eval_batch_size=120,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [14]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    for k in [1, 2, 3, 4,5]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
1,0.138575,0.079730,0.742740,0.652431,0.694662,0.441090,0.774916,0.562181,0.313887,0.827164,0.455082,0.244825,0.860227,0.381168,0.201132,0.883382,0.327661
2,0.074103,0.067660,0.776560,0.682138,0.726293,0.462730,0.812934,0.589762,0.328240,0.864988,0.475892,0.254820,0.895346,0.396729,0.208500,0.915743,0.339664


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [16]:
from pathlib import Path
import shutil
import re
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = Path("/kaggle/working/deberta_pair_cls")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_step(path):
    match = re.search(r"checkpoint-(\d+)", path.name)
    return int(match.group(1)) if match else -1

input_checkpoints = [
    p for p in Path("/kaggle/input").rglob("checkpoint-*")
    if p.is_dir()
]

input_checkpoints = sorted(input_checkpoints, key=checkpoint_step)

if len(input_checkpoints) == 0:
    print("No checkpoints found in /kaggle/input.")
    print("Training will start from scratch.")
else:
    print(f"Found {len(input_checkpoints)} checkpoint(s) in /kaggle/input:")

    for p in input_checkpoints:
        print(" -", p)

    print("\nCopying checkpoints to:", OUTPUT_DIR)

    for ckpt in input_checkpoints:
        dst = OUTPUT_DIR / ckpt.name

        if dst.exists():
            print(f"Already exists, skipping: {dst}")
            continue

        shutil.copytree(ckpt, dst)
        print(f"Copied: {ckpt.name}")

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
if last_checkpoint is None:
    print("\nNo checkpoint available in OUTPUT_DIR.")
    print("Next training cell should start from scratch.")
else:
    print("\nLast checkpoint available:")
    print(last_checkpoint)
    print("Next training cell should resume from this checkpoint.")

Found 2 checkpoint(s) in /kaggle/input:
 - /kaggle/input/notebooks/mikeelio4/model1/distilbert_results/checkpoint-2709
 - /kaggle/input/notebooks/mikeelio4/model1/distilbert_results/checkpoint-5418

Copying checkpoints to: /kaggle/working/deberta_pair_cls
Copied: checkpoint-2709
Copied: checkpoint-5418

Last checkpoint available:
/kaggle/working/deberta_pair_cls/checkpoint-5418
Next training cell should resume from this checkpoint.


In [17]:
if last_checkpoint is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Epoch,Training Loss,Validation Loss,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
3,0.066233,0.064173,0.787060,0.691362,0.736114,0.468460,0.823000,0.597065,0.331660,0.874001,0.480850,0.257505,0.904780,0.400909,0.210356,0.923895,0.342688
4,0.062648,0.062647,0.791320,0.695104,0.740098,0.471400,0.828165,0.600812,0.333493,0.878832,0.483508,0.258505,0.908294,0.402466,0.211208,0.927637,0.344076
5,0.060715,0.062099,0.792880,0.696474,0.741557,0.472770,0.830572,0.602558,0.334227,0.880765,0.484571,0.259025,0.910121,0.403276,0.211572,0.929235,0.344668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [18]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation Results: {'eval_loss': 0.06209854409098625, 'eval_precision_at_1': 0.79288, 'eval_recall_at_1': 0.6964740605400467, 'eval_f1_at_1': 0.7415568503848636, 'eval_precision_at_2': 0.47277, 'eval_recall_at_2': 0.8305721965531174, 'eval_f1_at_2': 0.6025579750320225, 'eval_precision_at_3': 0.33422666666666667, 'eval_recall_at_3': 0.8807645684369565, 'eval_f1_at_3': 0.4845714064788011, 'eval_precision_at_4': 0.259025, 'eval_recall_at_4': 0.9101210449570457, 'eval_f1_at_4': 0.4032757151030862, 'eval_precision_at_5': 0.211572, 'eval_recall_at_5': 0.9292352558809578, 'eval_f1_at_5': 0.34466849775675173, 'eval_runtime': 86.3559, 'eval_samples_per_second': 579.0, 'eval_steps_per_second': 2.42, 'epoch': 5.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.06262536346912384, 'eval_precision_at_1': 0.78988, 'eval_recall_at_1': 0.6929258193557443, 'eval_f1_at_1': 0.7382332049796254, 'eval_precision_at_2': 0.4725, 'eval_recall_at_2': 0.8290055442487192, 'eval_f1_at_2': 0.6019261637239165, 'eval_precision_at_3': 0.3342, 'eval_recall_at_3': 0.8795354059934031, 'eval_f1_at_3': 0.4843571856460994, 'eval_precision_at_4': 0.259025, 'eval_recall_at_4': 0.9089234332233841, 'eval_f1_at_4': 0.40315802580584914, 'eval_precision_at_5': 0.2118, 'eval_recall_at_5': 0.9290125622850727, 'eval_f1_at_5': 0.3449556346011023, 'eval_runtime': 86.1764, 'eval_samples_per_second': 580.206, 'eval_steps_per_second': 2.425, 'epoch': 5.0}
